# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
# The dataset contains mass spectrometry data on human extracellular vesicles proteins

# Define the dataset Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata.to_json()
print(f"{getattr(dataset.metadata, 'name', '')}\n{getattr(dataset.metadata, 'description', '')}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

We'll print all available RecordSets (`@id`), their descriptive names (if present), and their contained fields (`@id` and `name`).

In [ ]:
# Explore record sets, fields, and columns
from pprint import pprint

def print_record_sets_info(ds):
    # List all available record sets and their fields
    record_sets = list(ds.record_sets)
    record_set_ids = []
    print("Available Record Sets:")
    for rs in record_sets:
        rs_id = getattr(rs, '@id', None)
        rs_name = getattr(rs, 'name', None)
        print(f"- @id: {rs_id}")
        print(f"  Name: {rs_name}")
        record_set_ids.append(rs_id)
        # Print their fields
        if hasattr(rs, 'fields') and rs.fields:
            print("  Fields:")
            for f in rs.fields:
                f_id = getattr(f, '@id', None)
                f_name = getattr(f, 'name', None)
                print(f"    - @id: {f_id}  Name: {f_name}")
        print()
    return record_set_ids

all_record_set_ids = print_record_sets_info(dataset)

# Show 2 sample records from each discovered RecordSet
for rs_id in all_record_set_ids:
    print(f"First 2 records from RecordSet: {rs_id}")
    for i, rec in enumerate(dataset.records(record_set=rs_id)):
        pprint(rec)
        if i > 0:
            break

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis.
We will extract all available record sets (using their `@id`s) into Pandas DataFrames for exploration. The field names/IDs discovered previously will be used for reference.

In [ ]:
# Extract data from each record set (@id)
record_sets = all_record_set_ids  # as discovered earlier
dataframes = {}
for record_set in record_sets:
    print(f"Loading RecordSet: {record_set}")
    records = list(dataset.records(record_set=record_set))
    if records:
        dataframes[record_set] = pd.DataFrame(records)
        print(f"Fields/columns: {dataframes[record_set].columns.tolist()}")
        display(dataframes[record_set].head())
    else:
        print("No records found.")

# Pick the first non-empty RecordSet for further analysis
main_rs_id = None
for k, v in dataframes.items():
    if not v.empty:
        main_rs_id = k
        break

if main_rs_id:
    print(f"Main RecordSet for analysis: {main_rs_id}")
    print("Columns:", dataframes[main_rs_id].columns.tolist())
    dataframes[main_rs_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records, normalizing numeric fields, and grouping data. The following demonstrates filtering and normalization using numeric fields, all field references by their `@id`.

In [ ]:
# Please update the following IDs after inspecting columns above!
# For demonstration, let's try with some likely numeric fields (common for mass spec datasets):
# e.g. 'cr:coverage' or 'cr:molecular_weight' or similar (adapt or replace by your dataset fields)

if main_rs_id:
    cols = dataframes[main_rs_id].columns
    # Pick the first numeric field for demonstration
    # If not known, replace this selection by the exact @id
    numeric_field = None
    candidate_fields = [c for c in cols if ('coverage' in c or 'count' in c or 'abundance' in c or 'weight' in c or 'peptide' in c or 'intensity' in c)]
    # Find first numeric candidate
    for f in candidate_fields:
        try:
            dataframes[main_rs_id][f] = pd.to_numeric(dataframes[main_rs_id][f])
            if dataframes[main_rs_id][f].notna().any():
                numeric_field = f
                break
        except Exception:
            continue
    if not numeric_field:
        # fallback to first float-type column
        for f in cols:
            try:
                dataframes[main_rs_id][f] = pd.to_numeric(dataframes[main_rs_id][f])
                if dataframes[main_rs_id][f].dtype in [float, int]:
                    numeric_field = f
                    break
            except Exception:
                continue
    print(f"Using numeric field for filtering/normalization: {numeric_field}")
    df_main = dataframes[main_rs_id]
    if numeric_field and (numeric_field in df_main.columns):
        # Drop NaNs for computation
        df_main_num = df_main.dropna(subset=[numeric_field])
        threshold = df_main_num[numeric_field].mean()  # e.g. mean as threshold
        filtered_df = df_main_num[df_main_num[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold:.2f} (mean): {len(filtered_df)} found.")
        print(filtered_df[[numeric_field]].head())

        filtered_df[f"{numeric_field}_normalized"] = (
            (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        )
        print(f"Normalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Attempt to group by another attribute, e.g. sample, accession, or any category
        candidate_group_fields = [c for c in cols if ('sample' in c or 'type' in c or 'protein' in c or 'accession' in c or 'category' in c)]
        group_field = candidate_group_fields[0] if candidate_group_fields else None
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame()
            print(f"Grouped data by {group_field} (mean {numeric_field}):")
            print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. For example, plot a histogram of the main numeric field, and a boxplot grouped by a categorical/group field. 

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_rs_id and numeric_field:
    df_main = dataframes[main_rs_id]
    plt.figure(figsize=(8,4))
    sns.histplot(df_main[numeric_field].dropna(), bins=40, kde=True, color='skyblue')
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()

    # Boxplot by group
    group_field = None
    candidate_group_fields = [c for c in df_main.columns if ('sample' in c or 'type' in c or 'protein' in c or 'accession' in c or 'category' in c)]
    if candidate_group_fields:
        group_field = candidate_group_fields[0]
        plt.figure(figsize=(10,5))
        sns.boxplot(x=group_field, y=numeric_field, data=df_main)
        plt.title(f"{numeric_field} by {group_field}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
In this notebook, we demonstrated how to load, inspect, and analyze mass spectrometry proteomics data using the `mlcroissant` library. We explored the Croissant schema, programmatically discovered record sets and fields via their `@id`, filtered and normalized numeric data, and visualized key distributions. You can adapt the EDA and visualization cells by referring to field and record set `@id`s discovered for your specific application.